In [ ]:
PROJECT_DIR = '/nasdata/archive/personal_project_archives/bule/berner_daten_analyse'

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

In [ ]:
pd.read_csv('/nasdata/archive/personal_project_archives/bule/berner_daten_analyse/data/clinical/baseline_bern_test_impute_10n.csv')

In [ ]:
segmentation_mask_path= os.path.join(PROJECT_DIR,"data","patient_results")

In [ ]:
fname = os.path.join(PROJECT_DIR, 'data', 'patient_results', '76813', 'lesion_msk.nii.gz')
img = nib.load(fname)
data = img.get_fdata()
data

In [ ]:
# read image tensor 
'/nasdata/archive/personal_project_archives/bule/berner_daten_analyse/data/npz_output_tensors/413x128x128x28_dwi_bet_normalized_trilinear.npz'

In [ ]:
import numpy as np

## lesion volume
lesion_volume_df={"Patient_ID":[],"Lesion_volume_mm3":[],"Lesion_volume_ml":[]}
for path in [os.path.join(segmentation_mask_path,path,"lesion_msk.nii.gz") for path in os.listdir(segmentation_mask_path)]:
    


### dwi tensorf and dataframe
path = '/home/bule/xAI_bern/data/1030x128x128x28_dwi_bet_normalized_trilinear_bern_c_soop.npz'
npz = np.load(path, allow_pickle=True)
tensor_data = npz['tensor'].astype(np.float32)
df = pd.DataFrame(npz['df'].item())


In [ ]:
segmentation_mask_path= os.path.join(PROJECT_DIR,"data","patient_results")
p_id_list=os.listdir(segmentation_mask_path)


def load_mask(patient_id):
    mask_path = os.path.join(segmentation_mask_path, patient_id, 'lesion_msk.nii.gz')
    print(f"Loading mask from: {mask_path}")
    if os.path.exists(mask_path):
        return nib.load(mask_path).get_fdata()
    return None

In [ ]:
import ipywidgets as widgets

# assume p_id_list is defined, e.g.:
# p_id_list = sorted(df['p_id'].astype(str).unique())

p_id_dropdown = widgets.Dropdown(
    options=p_id_list,
    description='p_id',
    layout={'width': '300px'}
)
slice_slider = widgets.IntSlider(
    value=14,
    min=0,
    max=tensor_data.shape[3] - 1,
    step=1,
    description='Slice'
)

def show_patients(p_id, slice_idx=14):
    idx_arr = df[df['p_id'] == float(p_id)]['p_idx'].values
    if len(idx_arr) == 0:
        print(f"p_id {p_id} not found in df")
        return
    tensor = tensor_data[idx_arr[0]]
    print(tensor.shape)
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(tensor[:, :, slice_idx], cmap='gray')
    plt.title(f'DWI | Patient: {p_id} | Slice: {slice_idx}')
    plt.axis('off')

    mask = load_mask(str(p_id))
    
    print(mask.shape)
    plt.subplot(1, 2, 2)
    if mask is not None:
        plt.imshow(np.flipud(mask[:, :, slice_idx]), cmap='Reds')
        plt.title('Lesion Mask')
    else:
        plt.imshow(np.zeros_like(tensor[:, :, slice_idx]), cmap='gray')
        plt.title('No Mask Found')
    plt.axis('off')

    plt.show()

widgets.interactive(show_patients, p_id=p_id_dropdown, slice_idx=slice_slider)
